# Algoritmo de BackPropagation multiclase

Este codigo se utilizará para entrenar la red neuronal de clasificacion de Nuestras Caras



### Código en Python del Algoritmo de BackPropagation

La clase del dataset debe ser categorica
Existe una clase en Python que resuelve el problema

In [ ]:
# conexion al Google Drive
from google.colab import drive
drive.mount('/content/.drive')
!mkdir -p "/content/.drive/My Drive/DMA"
!mkdir -p "/content/buckets"
!ln -s "/content/.drive/My Drive/DMA" /content/buckets/b1

In [ ]:
# instalo  itables solo si no esta instalado
!pip show itables >/dev/null || pip install itables

In [ ]:
import polars as pl
import numpy as np
import math
import matplotlib.pyplot as plt
%matplotlib inline
from IPython import display
import time
import os
import pickle
from functools import reduce
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)


In [ ]:
# definicion de la clase de graficos

class perceptron_plot:
    """plotting first hidden layer class"""
    def __init__(self, X, Y, delay) -> None:
        # Guarda los atributos de la entrada
        self.X = X
        self.Y = Y
        self.delay = delay
        # Calculo de los limites del grafico
        x1_min = np.min(X[:,0])
        x2_min = np.min(X[:,1])
        x1_max = np.max(X[:,0])
        x2_max = np.max(X[:,1])
        # Se amplian los limites del grafico con 10% extra al mín y máx de c/eje
        self.x1_min = x1_min - 0.1*(x1_max - x1_min)
        self.x1_max = x1_max + 0.1*(x1_max - x1_min)
        self.x2_min = x2_min - 0.1*(x2_max - x2_min)
        self.x2_max = x2_max + 0.1*(x2_max - x2_min)
        # Configuración del gráfico: tamaño 10x8 y crea un subgráfico (ax)
        self.fig = plt.figure(figsize = (10,8))
        self.ax = self.fig.subplots()
        # Se establecen límites del gráfico
        self.ax.set_xlim(self.x1_min, self.x1_max, auto=False)
        self.ax.set_ylim(self.x2_min, self.x2_max, auto=False)

    def graficarVarias(self, W, x0, epoch, error) -> None:
        # Limpia y prepara el gráfico
        display.clear_output(wait =True)
        plt.cla()

        # Reconfigura los ejes y título
        self.ax.set_xlim(self.x1_min, self.x1_max)
        self.ax.set_ylim(self.x2_min, self.x2_max)
        plt.title( 'epoch ' + str(epoch) + '  reg ' + "{0:.2E}".format(error))
        # Ploteo de puntos
        num_classes = len(np.unique(self.Y))
        scatter = self.ax.scatter(self.X[:,0], self.X[:,1], c=self.Y, s=20)

        # Dibuja las rectas de desicion
          # Para cada neurona (fila de W) calcula la recta en func de la ecuac del plano
          # Dibuja esta línea desde x1_min hasta x1_max.
        for i in range(len(x0)):
            vx2_min = -(W[i,0]*self.x1_min + x0[i])/W[i,1]
            vx2_max = -(W[i,0]*self.x1_max + x0[i])/W[i,1]

            self.ax.plot([self.x1_min, self.x1_max],
                         [vx2_min, vx2_max],
                         linewidth = 2,
                         color = 'red',
                         alpha = 0.5)

        # Muestra el gráfico actualizado
        display.display(plt.gcf())
        time.sleep(self.delay)


In [ ]:
# definicion de las funciones de activacion
#  y sus derivadas
#  ahora agregando las versiones VECTORIZADAS

def func_eval(fname, x):
    # Selecciona y evalúa la función de activación para un valor escalar x según el nombre fname, usando match case
    match fname:
        case "purelin":       # Lineal, sin activacion
            y = x
        case "logsig":        # Sigmoide logística
            y = 1.0 / ( 1.0 + math.exp(-x) )
        case "tansig":        # Tangente sigmoidal, rango [-1, 1]
            y = 2.0 / ( 1.0 + math.exp(-2.0*x) ) - 1.0
    return y

# Version vectorizada de func_eval:
func_eval_vec = np.vectorize(func_eval) # np.vectorize() permite aplicar func_eval a arrays de NumPy como si fuera un elemento por elemento.

def deriv_eval(fname, y):  #atencion que y es la entrada y=f( x )
# Calcula la derivada de la función de activación respecto a su entrada x, usando "y" como argumento.
  # fname: nombre de la función de activación.
  # y: salida de la función de activación (ya aplicada a x, es decir, y = f(x)).
    match fname:
        case "purelin":
            d = 1.0
        case "logsig":
            d = y*(1.0-y)
        case "tansig":
            d = 1.0 - y*y
    return d


# version vectorizada de deriv_eval
deriv_eval_vec = np.vectorize(deriv_eval)

### Clase  multiperceptron
entrenar, predecir

In [ ]:
# Definicion de la clase de multiperceptron (red neuronal con múltiples capas ocultas)
      # Es modular y persistente (guarda y carga modelos)

class multiperceptron(object):
    """Multiperceptron class"""

    # Inicializacion de los pesos de todas las capas

    def _red_init(self, semilla) -> None:
      # Este método se encarga de inicializar los pesos y sesgos de todas las capas de la red neuronal.
      # También configura propiedades específicas de cada capa (como la cantidad de neuronas, si es la última capa, y la función de activación usada).
      # Usa semilla aleatoria para reproducibilidad.
      # Para cada capa definida: 1) Crea una estructura nivel con; -Tamaño de entrada y salida. -Función de activación(func) -Pesos W (matriz de conexiones). -Sesgos w0. -Variables de momento (W_m, w0_m) para optimización.
      #                          2) Agrega nivel a la lista de capas en self.red['layer'].

        niveles = self.red['arq']['layers_qty'] # Obtiene la cantidad de capas ocultas + salida (layers_qty incluye todas las capas internas de la red - NO incluye la capa de entrada).

        np.random.seed(semilla)

        for i in range(niveles):
           nivel = dict()          # Se crea un diccionario nivel donde se guardará toda la información de una capa individual: pesos, función de activación, etc.
           nivel['id'] = i                                    # Se guarda el índice de la capa (0, 1, 2, …).
           nivel['last'] = (i==(niveles-1))                   # Indica si es la última capa de la red. (útil luego para distinguir entre capa oculta y capa de salida).
           nivel['size'] = self.red["arq"]["layers_size"][i]  # Consulta la arquitectura ('arq') de la red, y obtiene el número de neuronas que debe tener esta capa.
           nivel['func'] = self.red["arq"]["layers_func"][i]  # Consulta la arquitectura ('arq') de la red, y obtiene la func de activación asignada a la capa.

           # Se determina cuántas entradas recibe la capa actual
           if( i==0 ):
              entrada_size = self.red['arq']['input_size']          # Si es la primera capa las entradas vienen directamente del input del dataset.
           else:
              entrada_size =  self.red['arq']['layers_size'][i-1]   # Si no, las entradas vienen de la capa anterior, así que se toma el tamaño de la capa anterior

           salida_size =  nivel['size']                             # El tamaño de salida de la capa es la cant de neuronas que contiene (c/neurona genera una salida).

           # los pesos, inicializados random
           nivel['W'] = np.random.uniform(-0.5, 0.5, [salida_size, entrada_size]) # Inicializa los pesos con valores aleatorios en  [-0.5, 0.5].
           nivel['w0'] = np.random.uniform(-0.5, 0.5, [salida_size, 1])           # Inicialización del vector de sesgos de cada beurona

           # los momentos, inicializados en CERO
           nivel['W_m'] = np.zeros([salida_size, entrada_size])     # Son variables auxiliares usadas para el término de momento en el entrenamiento.
           nivel['w0_m'] = np.zeros([salida_size, 1])

           self.red['layer'].append(nivel)                          # Guarda capa configurada: Añade el dicc "nivel" (con pesos, sesgos, momentos, etc.) a la lista de capas de la red.

    # constructor generico
    def __init__(self) -> None:
        self.data = dict()                      # donde se guardarán los datos de entrada, etiquetas y codificaciones.
        self.red = dict()                       # donde se almacenará la arquitectura, pesos, func de activación y demas datos internos de la red.
        self.carpeta = ""                       # carpeta en disco donde se guarda el modelo entrenado


    # inicializacion full
    def inicializar(self, df, campos, clase, hidden_layers_sizes, layers_func,
                 semilla, carpeta) -> None:

    # Parámetros clave: df:dataset - campos:columnas de entrada - clase:columna de salida - hidden_layers_sizes:lista con cantidad de neuronas por capa oculta -
    #                   layers_func:funciones de activación por capa - semilla:para inicializar pesos - carpeta:dónde guardar modelo.


        # Procesamiento de los datos de entrada (features)
        self.data['X'] = np.array(df.select(campos))                # Convierte las columnas de entrada "campos" a un array NumPy y lo guarda como X en self.data (atributo llamado "data" que pertenece a esa instancia))
        # Normalización (escalado) de los datos para que los atributos tengan media 0 y desviación estándar 1
        X_mean = self.data['X'].mean(axis=0)
        X_sd = self.data['X'].std(axis=0)
        self.data['X'] = (self.data['X'] - X_mean)/X_sd

        #  Procesamiento de etiquetas Ylabel en  numpy
        label =df.select(clase)                                     # Extrae la columna clase como etiquetas originales.
        self.data['Ylabel'] = np.array(label).reshape(len(label))   # Se guarda como vector 1D en self.data['Ylabel']

        # one-hot-encoding de Y . Realiza la codificacion "one hot" de las etiquetas (Convierte cada categoría única en una columna binaria separada y representa la presencia con 1 y la ausencia con 0.)
        col_originales = df.columns
        self.data['Y'] = np.array( df.to_dummies(clase).drop(col_originales, strict=False) )
        col_dummies = sorted( list( set(df.to_dummies(clase).columns) -  set(col_originales)))
        clases_originales = reduce(lambda acc, x: acc + [x[(len(clase)+1):]], col_dummies, [])

        # Construcción de la arquitectura de la red
        tamanos = hidden_layers_sizes                               # Copia los tamaños de capas ocultas.
        tamanos.append(self.data['Y'].shape[1])                     # Añade al final la cantidad de neuronas de salida = cantidad de clases (n_clases).

        arquitectura = {                                   # Detalle de este diccionario:
             'input_size' : self.data['X'].shape[1],                # Número de atributos de entrada
             'input_mean' : X_mean,                                 # Guardado para escalar futuros datos en predicción
             'input_sd' :  X_sd,
             'output_values' : clases_originales,                   # Lista con los nombres de las clases
             'layers_qty' : len(hidden_layers_sizes),               # Número de capas ocultas + salida  (NO INCLUYE la de entrada)
             'layers_size' : tamanos ,                              # Lista con el tamaño de cada capa (ocultas + salida)
             'layers_func' : layers_func,                           # Lista de fuciones de activacion para cada capa
        }

        self.red['arq'] = arquitectura                              # Se guarda esta arquitectura en el atributo self.red


        # inicializo  work . Inicialización de información de entrenamiento
        self.red['work'] = dict()
        self.red['work']['epoch'] = 0                               # contador de Epochs
        self.red['work']['MSE'] = float('inf')                      # error cuadrático medio inicial = infinito
        self.red['work']['train_error_rate'] = float('inf')         # tasa de error de clasificación (inicialmente infinita).

        # Preparación de las capas y pesos
        self.red['layer'] = list()                                  # Se inicializa self.red['layer'] como lista vacía.
        self._red_init(semilla)                                     # Se llama a _red_init, que construye e inicializa los pesos y sesgos para cada capa

        # grabo el entorno. Guardar el entorno en disco
        self.carpeta = carpeta
        os.makedirs(self.carpeta, exist_ok=True)
        with open(self.carpeta+"/data.pkl", 'wb') as f:             # Guarda los datos (X, Y, Ylabel, etc.) en data.pkl
            pickle.dump(self.data, f)

        with open(self.carpeta+"/red.pkl", 'wb') as f:              # Guarda la red neuronal (arq, layer, work) en red.pkl
            pickle.dump(self.red, f)





    # Algoritmo Backpropagation
    def  entrenar(self, epoch_limit, MSE_umbral,
               learning_rate, lr_momento, save_frequency,
               retomar=True) -> None:
    # Parametros clave: epoch_limit:número máximo de epochs (iteraciones completas sobre los datos) - MSE_umbral:valor mínimo de error cuadrático medio (criterio de parada) - learning_rate: tasa de aprendizaje.
    #                   lr_momento: coeficiente de momentum (suaviza las actualizaciones) - save_frequency: cada cuanto epochs guardar y graficar - retomar:si True, carga un modelo ya guardado previamente.
        # si debo retomar
        if( retomar):                                               # Cargar modelo previo (si corresponde)
            with open(self.carpeta+"/data.pkl", 'rb') as f:         # se cargan los archivos guardados previamente (self.data y self.red) desde el disco.
              self.data = pickle.load(f)
            with open(self.carpeta+"/red.pkl", 'rb') as f:
              self.red = pickle.load(f)


        # inicializaciones del bucle principal del backpropagation
        epoch = self.red['work']['epoch']                           # Inicialización de variables de entrenamiento, (Epochs y MSE se inicializan a partir del estado guardado).
        MSE = self.red['work']['MSE']

        # inicializacion del grafico de evolucion
        grafico = perceptron_plot(X=self.data['X'], Y=self.data['Ylabel'], delay=0.1)   # Se crea una instancia (de perceptron_plot) para graficar la evolución de la primera capa oculta.

        # Parametros de control
        Xfilas = self.data['X'].shape[0]                            # cantidad de registros de entrada
        niveles = self.red["arq"]["layers_qty"]                     # cantidad de capas ocultas + capa de salida (sin contar la de entrada).

        while ( MSE > MSE_umbral) and (epoch < epoch_limit) :       # continuo mientras error cuadratico medio muy grande  y NO llegué al límite de epochs
          epoch += 1


          # Entrenamiento Registro por Registro.
          # recorro siempre TODOS los registros de entrada
          for fila in range(Xfilas):
             # fila es el registro actual
             x = self.data['X'][fila:fila+1,:]
             clase = self.data['Y'][fila:fila+1,:]                  # Se obtiene un registro de entrada x y su etiqueta clase.


             # FORWARD PROPAGATION: propagar el x hacia adelante.
             # Se inicializa la entrada y una lista para guardar la salida de cada capa
             entrada = x.T                  # la entrada a la red
             vsalida =  [0] *(niveles)      # salida de cada nivel de la red
             # etapa forward - recorro hacia adelante, nivel a nivel
             for i in range(niveles):                                                       # Por cada capa:
               estimulos = self.red['layer'][i]['W'] @ entrada + self.red['layer'][i]['w0']       #Se calcula el estímulo total: W·x + b.
               vsalida[i] =  func_eval_vec(self.red['layer'][i]['func'], estimulos)               # Se aplica la función de activación.
               entrada = vsalida[i]  # para la proxima vuelta                                     # El resultado se convierte en la entrada para la próxima capa.



             # BACKPROPAGATION — cálculo de errores
             # etapa backward - calculo los errores en la capa hidden y la capa output
             verror =  [0] *(niveles+1)                      # inicializo dummy. Se inicializa una lista para errores.
             verror[niveles] = clase.T - vsalida[niveles-1]  #El error en la capa de salida es la diferencia entre el valor esperado y el obtenido.

             i = niveles-1
             verror[i] = verror[i+1] * deriv_eval_vec(self.red['layer'][i]['func'], vsalida[i]) # Se calcula el gradiente de la capa de salida (derivada del error respecto de la salida).

             for i in reversed(range(niveles-1)):                                                                     # Para cada capa anterior:
               verror[i] = deriv_eval_vec(self.red['layer'][i]['func'], vsalida[i])*(self.red['layer'][i+1]['W'].T @ verror[i+1])      # Se propaga el error hacia atrás usando la derivada de la func de activación y los pesos de la siguiente capa

             # Actualización de pesos y sesgos:
             # ya tengo los errores que comete cada capa
             # corregir matrices de pesos, voy hacia atras
             # backpropagation
             entrada = x.T
             for i in range(niveles):                                                                                 # Se actualizan los "momentum" para pesos y sesgos coN Δpeso = tasa X gradiente + momentum X Δpeso_anterior.
               self.red['layer'][i]['W_m'] = learning_rate *(verror[i] @ entrada.T) + lr_momento *self.red['layer'][i]['W_m']
               self.red['layer'][i]['w0_m'] = learning_rate * verror[i] + lr_momento * self.red['layer'][i]['w0_m']

               self.red['layer'][i]['W']  =  self.red['layer'][i]['W'] + self.red['layer'][i]['W_m']                  # Se ajustan los pesos reales y se propaga la entrada para la siguiente capa
               self.red['layer'][i]['w0'] =  self.red['layer'][i]['w0'] + self.red['layer'][i]['w0_m']
               entrada = vsalida[i]  # para la proxima vuelta


          # Valuación del modelo después de un Epoch
          # ya recalcule las matrices de pesos, ahora avanzo la red, feed-forward
          # para calcular el red(X) = Y
          entrada = self.data['X'].T
          for i in range(niveles):                                                          # Se calcula la salida del modelo en modo batch, para todo X (propagación completa).
            estimulos = self.red['layer'][i]['W'] @ entrada + self.red['layer'][i]['w0']
            salida =  func_eval_vec(self.red['layer'][i]['func'], estimulos)
            entrada = salida  # para la proxima vuelta

          # calculo el error cuadratico medio TODOS los X del dataset
          MSE= np.mean( (self.data['Y'].T - salida)**2 )                                    # Se calcula el Error Cuadrático Medio (MSE) en el conjunto completo.



          # Visualización y guardado periódico: cada "save_frequency" Epochs (o al finalizar)
          # Grafico las rectas SOLAMENTE de la Primera Hidden Layer
          # tengo que hacer w0.T[0]  para que pase el vector limpio
          if( epoch % save_frequency == 0 ) or ( MSE <= MSE_umbral) or (epoch >= epoch_limit) :
              # grafico
              W = self.red['layer'][0]['W']
              w0 = self.red['layer'][0]['w0']
              grafico.graficarVarias(W, w0.T[0], epoch, MSE)
              # almaceno en work
              self.red['work']['epoch'] = epoch                     # Se actualizan las estadísticas del entrenamiento
              self.red['work']['MSE'] = MSE
              prediccion = np.argmax( salida.T, axis=1)             # Se genera la presiccion final por clase
              # prediccion
              out = np.array(self.red["arq"]['output_values'])
              error_rate = np.mean( self.data['Ylabel'] != out[prediccion])
              self.red["work"]['train_error_rate'] = error_rate # error_rate != error cuadratico medio   # Se calcula la tasa de error clasificatoria sobre el conjunto de entrenamiento


              # grabo a un archivo la red neuronal ENTRENADA por donde esté
              #   solo la red, NO los datos
              with open(carpeta+"/red.pkl", 'wb') as f:
                 pickle.dump(self.red, f)

        return (epoch, MSE, self.red['work']['train_error_rate'] )    # Devuelve la cantidad de Epochs realizados, el MSE final y el error clasificatorio final.


    # predigo a partir de modelo recien entrenado
    def  predecir(self, df_new, campos, clase) -> None:
    # Parametros claves: df_new:nuevo conj de datos sobre el cual predecir - campos: lista de nombres de las columnas con las variables independientes (inputs) - clase: nombre de la columna con la variable objetivo (solo usada para evaluar la predicción).

        niveles = self.red['arq']['layers_qty']                  # Se obtiene cuántas capas hay en total (ocultas + salida, sin incluir la de entrada).

        # etapa forward
        # recorro hacia adelante, nivel a nivel
        X_new =  np.array( df_new.select(campos))               # Se seleccionan los campos/columnas de interés del nuevo DataFrame y se convierten a un array de NumPy.


        # estandarizo manualmente con las medias y desvios que almacene durante el entrenamiento
        X_new = (X_new - self.red['arq']['input_mean'])/self.red['arq']['input_sd']   # Esto para asegurar que los datos nuevos estén en la misma escala.

        # grafico los datos nuevos
        Ylabel_new =df_new.select(clase)                                        #   Se obtiene la columna de clase esperada (clase) del DataFrame nuevo.
        Ylabel_new = np.array(Ylabel_new).reshape(len(Ylabel_new))              #   Se convierte a un array NumPy y se "aplana" para que sea un vector de etiquetas.

        grafico = perceptron_plot(X=X_new, Y=Ylabel_new, delay=0.1)
        W = self.red['layer'][0]['W']
        w0 = self.red['layer'][0]['w0']
        grafico.graficarVarias(W, w0.T[0], epoch, MSE)

        # la entrada a la red,  el X que es TODO  x_new
        entrada = X_new.T  # traspongo, necesito vectores columna

        for i in range(niveles):                                                      #Para cada capa:
          estimulos = self.red['layer'][i]['W'] @ entrada + self.red['layer'][i]['w0']        # Se calculan los estímulos netos (producto matriz de pesos + sesgos).
          salida =  func_eval_vec(self.red['layer'][i]['func'], estimulos)                    # Se aplica la función de activación correspondiente.
          entrada = salida  # para la proxima vuelta                                          # El resultado se convierte en la entrada para la siguiente capa. Al final de este bucle, "salida" representa la salida final de la red

        # me quedo con la neurona de la ultima capa que se activio con mayor intensidad
        pred_idx = np.argmax( salida.T, axis=1)                                   # Para cada muestra, selecciona el índice de la neurona de salida más activada (es decir, la predicción de clase).
        pred_raw = np.max( salida.T, axis=1)                                      # Obtiene el valor de activación más alto (confianza) para cada muestra

        # calculo error_rate
        out = np.array(self.red['arq']['output_values'])                          # Se recupera el vector con los nombres reales de las clases (output_values).
        error_rate = np.mean(np.array(df_new.select("y") != out[pred_idx]))       # Se compara la clase predicha con la etiqueta real (df_new.select("y")) y calcula la tasa de error de clasificación (proporción de predicciones incorrectas).

        return (out[pred_idx], pred_raw, error_rate)                              # Devuelve: out[pred_idx]:clases predichas como etiquetas (no índices) - pred_raw:nivel de activación máxima por muestra (confianza) - error_rate: tasa de error (porcentaje de predicciones fallidas).


    # cargo un modelo ya entrenado, grabado en carpeta
    # Permite recuperar una red entrenada previamente sin tener que volver a procesar los datos o realizar el entrenamiento.
    # Carga el modelo desde disco, lo vuelve a tener disponible en memoria (self.red) y devuelve su estado de entrenamiento.
    def cargar_modelo(self, carpeta) -> None:
        self.carpeta = carpeta

        with open(self.carpeta+"/red.pkl", 'rb') as f:      # Se abre el archivo red.pkl, en modo lectura binaria ('rb').
          self.red = pickle.load(f)                         # Este objeto self.red contiene: arquitectura de la red, pesos y sesgos por capa, func de activación, estadísticas de entrenamiento (epoch, MSE, etc).

        return (self.red['work']['epoch'],
                self.red['work']['MSE'],
                self.red['work']['train_error_rate'] )


## 1 Lectura del Dataset

En este humilde y restringida version, la clase del dataset debe ser categorica, no es capaz de trabajar con clases continuas.
La clase categorica puede ser  n-aria

In [ ]:
# Lectura del dataset con la moderna libreria Polars  (Pandas debe extinguirse!)

df = pl.read_csv('https://storage.googleapis.com/open-courses/austral2025-af91/clusters02.txt', separator='\t')
df

### 1.1 Particion  training/testing



*   Es valido cambiar la *semilla_particion* para probar distintos <test, train> y asi estimar con mas precisión el error rate en testing  (Montecarlo Estimation)



In [ ]:
# particion del dataset en training/testing

semilla_particion = 12011977
pct_train = 0.75  # ratio de registros que va a training


def train_test_split_df(df, seed=0, test_size=0.2):
    return df.with_columns(
        pl.int_range(pl.len(), dtype=pl.UInt32)
        .shuffle(seed=seed)
        .gt(pl.len() * test_size)
        .alias("split")                                          # Añade la columna booleana al DataFrame original. Se nombra "split" a esta columna booleana que indica si una fila va a entrenamiento (True) o test (False).
    ).partition_by("split", include_key=False)                   # split=True → entrenamiento


(df_train, df_test) = train_test_split_df(df,                       # llama a la función y almacena los resultados
                                          seed=semilla_particion,
                                          test_size=pct_train)

# imprimo los tamaños
print("Train:", df_train.shape)
print("Test:", df_test.shape)

## 2  Entrenamiento del modelo

### 2.1  Inicializacion de la neural network



*   Es valido cambiar la *semilla_red* para arrancar el entrenamiento con distintas rectas iniciales


In [ ]:
# defino la red multiperceptron
carpeta = "/content/buckets/b1/nn01/"  # cambiar con cada corrida
semilla_red = 12011977  # define las rectas iniciales

# una sola capa oculta de 2 neuronas  [2]
# la capa oculta y la final tienen ambas logsig de activacion
mp = multiperceptron()
mp.inicializar(
    df=df_train, campos=["x1","x2"], clase="y",  # especificaion del dataset
    hidden_layers_sizes=[4],  # no va la capa final, solo hidden layers
    layers_func=['logsig','logsig'], # funciones de activacion de cada capa
    semilla=semilla_red,
    carpeta = carpeta
    )

### 2.2 Entrenamiento de la neural network = backpropagation

Aqui se hace el trabajo pesado de entrenar la red neuronal

Es necesario experimentar con


*   learning_rate
*   lr_momento
*   epoch_limit  y MSE_umbral



In [ ]:
# entreno la neural netowork con BackPropagation

# el entrenamiento
(epoch, MSE, train_error_rate) = mp.entrenar(           # Se llama al método entrenar() sobre el objeto mp, se espera devuelva: epoch: número de epoch ejecutados en el entrenamiento - MSE: Error cuadrático medio final -
                                                        # train_error_rate: tasa de error final del modelo en el conjunto de entrenamiento (porcentaje de clasificaciones incorrectas).
    epoch_limit=1500,
    MSE_umbral=0.006,
    learning_rate=0.2,
    lr_momento=0.2,                                     # Término de momento(momentum): ayuda a estabilizar y acelerar el aprendizaje al considerar también el paso anterior (suaviza oscilaciones y ayuda a salir de mínimos locales)
    save_frequency=100,                                 # Cada cuántos Echs se guardará una copia del estado de la red entrenada y se actualizarán los gráficos.
    retomar=True)                                       # Si True, el entrenamiento continúa desde donde se dejó previamente (cargando archivos guardados data.pkl, red.pkl). Si False, comenzará desde cero.


#### Visualizacion de los resultados de la salida del entrenamiento de la red

In [ ]:
# las metricas basica de la red
print("epoch :", epoch)
print("MSE :", MSE)
print("train_error_rate :", train_error_rate)

In [ ]:
# la primera hidden layer
print("W :", mp.red["layer"][0]["W"])
print()
print("w0 :", mp.red["layer"][0]["w0"])

### 2.3 Entrenamiento en caso de retomar



*   Si se cortó el colab
*   Si quiero extender la corrida a mas epochs
*   Si quiero cambiar el learninh_rate
*   Si quiero cambiar el MSE_umbral



In [ ]:
(epoch, MSE, train_error_rate) = mp.entrenar(
    epoch_limit=1800, # aumento
    MSE_umbral=0.001,
    learning_rate=0.05,
    lr_momento=0.05,
    save_frequency=100,
    retomar=True)

Visualizacion de los resultados de salida de un post entrenamiento

In [ ]:
print("epoch :", epoch)
print("MSE :", MSE)
print("train_error_rate :", train_error_rate)

In [ ]:
# la primera hidden layer
print("W :", mp.red["layer"][0]["W"])
print()
print("w0 :", mp.red["layer"][0]["w0"])

## 3  Prediccion en los datos de Testing


Se muestran los datos de testing, que son distintos a los de training

### 3.1 Prediccion en caliente

In [ ]:
# aplico la red entrenada al dataset de testing


(y_hat,y_raw, test_error_rate) = mp.predecir(df_new=df_test, campos=['x1', 'x2'], clase='y') # se llama al método .predecir() del objeto mp, que es una instancia de la clase multiperceptron.

# Aplica la red neuronal a los datos contenidos en df_test.
# Los campos de entrada utilizados son x1 y x2.
# El campo de salida verdadera (la etiqueta real) es y.

# devuelve
# y_hat: Lista con las clases predichas por la red neuronal para cada muestra en df_test.
# y_raw: Valores crudos de salida de la última capa de la red (valores reales de activación antes de aplicar argmax). Sirven como "confianza" de la predicción.
# test_error_rate: Tasa de error (porcentaje de predicciones incorrectas) sobre el conjunto de prueba.


#### Visualizacion del error en testing

In [ ]:
print("error_rate (train, test): ",  train_error_rate, test_error_rate)    # Muestra las tasas de error de entrenamiento (train_error_rate, obtenido al finalizar el entrenamiento) y de prueba (test_error_rate, recién calculado).

# permite comparar el rendimiento del modelo en ambos conjuntos:
# Una alta diferencia podría indicar OVERFITTING.
# Una baja diferencia suele indicar buena generalización.

#### Visualizacion de la prediccion en testing

In [ ]:
tb_salida_test = pl.DataFrame( {"clase":df_test["y"], "pred":y_hat, "y_raw":y_raw })  # construye una nueva tabla (tb_salida_test) que contiene: "clase"	Clase verdadera (y) del conjunto de prueba.
                                                                                      # "pred"	Clase predicha por la red (y_hat).
                                                                                      # "y_raw"	Valor de activación de la neurona más activa (nivel de confianza o intensidad de la predicción).
tb_salida_test

## 4 Prediccion en datos NUEVOS


*   La red fue entrenada en el pasado, y se grabó al drive
*   Ya no esta disponible la sesion donde se entreno
*   No quiero volver a entrenar de cero

In [ ]:
# cargo datos NUEVOS
df_new = pl.read_csv('https://storage.googleapis.com/open-courses/austral2025-af91/nuevos02.txt', separator='\t')
df_new.shape

In [ ]:
# cargo modelo grabado y lo aplico a los datos nuevos

carpeta = "/content/buckets/b1/nn01/"  # la carpeta del modelo entrenado

mp_frio = multiperceptron()
(epoch, MSE, train_error_rate) = mp_frio.cargar_modelo(carpeta)

(y_hat, y_raw, new_error_rate) = mp_frio.predecir(df_new=df_new, campos=['x1', 'x2'], clase='y')

#### Visualizacion del error modeloa aplicado a datos nuevos

In [ ]:
print("error_rate (train, new): ",  train_error_rate, new_error_rate)

#### Visualizacion de la prediccion en datos nuevos

In [ ]:
tb_salida_new = pl.DataFrame( {"clase":df_new["y"], "pred":y_hat, "y_raw":y_raw })
tb_salida_new